<a href="https://colab.research.google.com/github/deltorobarba/science/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###### *⚙️ Setup and Environments*

In [ ]:
# @title Installs
############################################

!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk,evaluation]"
!pip install --upgrade --quiet google-cloud-secret-manager google-cloud-bigquery anthropic[vertex]
!pip install --quiet a2a-sdk google-adk
!pip install --quiet litellm mcp
!pip show google-adk litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 k

In [ ]:
# @title Imports
############################################

import os, asyncio, json, litellm, warnings, vertexai, google.adk, google.auth, google.auth.transport.requests, time
warnings.filterwarnings('ignore')
from google.cloud import storage
from google import genai
from google.genai import types
from google.colab import userdata
from google.genai.types import HttpOptions
from google.genai import types as genai_types
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, Agent
from google.adk.events import Event
from google.adk.tools import google_search, url_context, VertexAiSearchTool, load_memory
from google.adk.sessions import InMemorySessionService, VertexAiSessionService
from google.adk.memory import InMemoryMemoryService, VertexAiMemoryBankService
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from typing import List
import pandas as pd
from pydantic import BaseModel
from vertexai import Client, types, agent_engines
from vertexai.preview import reasoning_engines  # Deployment, Tracing and Telemetry
from google.genai.types import (
    CreateBatchJobConfig,
    CreateCachedContentConfig,
    EmbedContentConfig,
    FunctionDeclaration,
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
    Tool,)

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [ ]:
# @title Environmental Variables
############################################

# These tell the underlying google-genai client to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = "deltorobarba"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
# Required by LiteLLM for Vertex AI non-Gemini models
os.environ["VERTEXAI_PROJECT"] = "deltorobarba"
os.environ["VERTEXAI_LOCATION"] = "us-central1"
# Initialize Vertex AI client
client = vertexai.Client(project="deltorobarba", location="us-central1")
PROJECT_ID="deltorobarba"
LOCATION="us-central1"

In [ ]:
# @title Connect to Google Cloud Project
from google.colab import auth
auth.authenticate_user()

###### ✅ *Model Garden*

In [ ]:
# @title Google SDK

# https://github.com/se02035/local-llm-proxy it helps to spin up a litellm proxy with local ollama. also supports a public endpoint using ngrok

# Locations: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/locations
# Google Models: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models
# Versions: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions

# client = genai.Client(http_options=HttpOptions(api_version="v1")) --> only for Google models directly
# https://github.com/GoogleCloudPlatform/generative-ai/blob/main/sdk/intro_genai_sdk.ipynb
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/open-models/use-maas
from google import genai
from google.genai.types import GenerateContentConfig
from google.genai import types

PROJECT_ID="deltorobarba"
LOCATION = "global"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

gemini    = "google/gemini-3.1-pro-preview"
gemma     = "google/gemma-4-26b-a4b-it-maas"
grok      = "xai/grok-4.1-fast-reasoning"
deepseek  = "deepseek-ai/deepseek-v3.2-maas"
kimi      = "moonshotai/kimi-k2-thinking-maas"
qwen      = "qwen/qwen3-next-80b-a3b-thinking-maas"
minimax   = "minimaxai/minimax-m2-maas"
glm       = "zai-org/glm-5-maas"
llama     = "meta/llama-3.3-70b-instruct-maas"   # LOCATION = "us-central1"
claude    = "anthropic/claude-opus-4-7"

system_instruction = """You are a helpful machine learning advisor and answer briefly."""
prompt = """Hi, which LLM are you?"""

generate_content_config = types.GenerateContentConfig(
    temperature=0.4,
    top_p=0.95,
    top_k=20,
    candidate_count=1,
    seed=5,
    max_output_tokens=60,
    stop_sequences=["STOP!"],
    presence_penalty=0.0,
    frequency_penalty=0.0,
    system_instruction=system_instruction,)

response = client.models.generate_content(
    model=gemma,                       # <--- ⚠️ Define model here
    contents=prompt,
    config=generate_content_config,
)
print("✅ Success")
print(response.text)

✅ Success
I am a large language model, trained by Google.


In [ ]:
# @title OpenAI SDK
#################################################

from openai import OpenAI

PROJECT  = "deltorobarba"
LOCATION = "global"

# ⚠️ Accept User Agreements per Model!
grok = "xai/grok-4.1-fast-reasoning"        # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/partner-models/grok/grok-4-1-fast
gemma = "google/gemma-4-26b-a4b-it-maas"    # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/google/gemma-4-26b-a4b-it
gemini = "google/gemini-3.1-pro-preview"    # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/3-1-pro
deepseek = "deepseek-ai/deepseek-v3.2-maas" # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32

# Get access token from Application Default Credentials (run `gcloud auth application-default login` once beforehand)
credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())

# Point OpenAI SDK at Vertex's OpenAI-compatible endpoint
client = OpenAI(
    base_url=f"https://aiplatform.googleapis.com/v1/projects/{PROJECT}/locations/{LOCATION}/endpoints/openapi",
    api_key=credentials.token,
)

# Call selectec model like any OpenAI model
response = client.chat.completions.create(
    model=deepseek,
    messages=[{"role": "user", "content": "Hi, which LLM are you?"}],
    temperature=0.2,
)

print("✅ Success")
print(response.choices[0].message.content)

✅ Success
你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，具有以下特点：
- 完全免费使用
- 支持128K上下文长度
- 可以处理文件上传（图像、txt、pdf、ppt、word、excel等）
- 支持联网搜索（需要手动开启）
- 可通过官方应用商店下载App使用

虽然我不支持多模态识别，但可以读取上传文件中的文字信息来帮助你。我的知识截止到2024年7月，会尽我所能为你提供准确、有用的回答！

有什么问题我可以帮你解答吗？我很乐意为你提供帮助！✨


In [ ]:
# @title LiteLLM
#################################################

# DeepSeek V3.2 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32
deepseek = LiteLlm(
    model="vertex_ai/deepseek-ai/deepseek-v3.2-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# Kimi K2 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/kimi/kimi-k2-thinking
kimi = LiteLlm(
    model="vertex_ai/kimi/kimi-k2-thinking-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# Qwen 3 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/qwen/qwen3-next-thinking
qwen = LiteLlm(
    model="vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# MiniMax - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/minimax
minimax = LiteLlm(
    model="vertex_ai/minimaxai/minimax-m2-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# GLM 5 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/zaiorg/glm-5
glm = LiteLlm(
    model="vertex_ai/zai-org/glm-5-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# ⚠️ ----- Quick Test on Model Availability -----
deepseek = "vertex_ai/deepseek-ai/deepseek-v3.2-maas"     # ✅
kimi = "vertex_ai/moonshotai/kimi-k2-thinking-maas"       # ✅
qwen = "vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas"  # ✅
minimax = "vertex_ai/minimaxai/minimax-m2-maas"           # ✅
glm ="vertex_ai/zai-org/glm-5-maas"                       # ✅

project = "deltorobarba"
location = "global"
question = "Hi"

try:
    response = litellm.completion(
        model=deepseek,                                    # <--- ⚠️ Add model here
        messages=[{"role": "user", "content": question}],
        temperature=0.2,
        #max_tokens=264,
        vertex_project=project,
        vertex_location=location)
    print("✅ Success: The model is reachable!")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"❌ Failed: Connectivity or Model Name error: {e}")

✅ Success: The model is reachable!
你好！👋 很高兴见到你！

有什么我可以帮助你的吗？无论是回答问题、聊天、协助解决问题，还是其他任何事情，我都很乐意为你提供帮助！😊


In [ ]:
# @title Vertex AI - Deployment Recommendation
from vertexai import model_garden

# google/gemma-4-26B-A4B-it
# google/gemma-4-31B-it

# Initialize for your project/location
# model_id can be "google/gemma-4-26B-A4B-it" or "google/gemma-4-31B-it"
model = model_garden.OpenModel("google/gemma-4-26B-A4B-it")

# Get the recommended/verified deployment options
deploy_options = model.list_deploy_options()
print(deploy_options)

[model_display_name: "google/gemma-4-26B-A4B-it"
container_spec {
  image_uri: "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20251205_0916_RC01"
  args: "python"
  args: "-m"
  args: "vllm.entrypoints.api_server"
  args: "--host=0.0.0.0"
  args: "--port=8080"
  args: "--swap-space=16"
  args: "--model=google/gemma-4-26B-A4B-it"
  args: "--revision=7d4c97e54145f8ffd1a4dd1b4986a5015a517842"
  args: "--max-model-len=2048"
  args: "--gpu-memory-utilization=0.9"
  args: "--enforce-eager"
  args: "--tensor-parallel-size=1"
  args: "--enable-chunked-prefill"
  env {
    name: "DEPLOY_SOURCE"
    value: "UI_HF_VERIFIED_MODEL"
  }
  env {
    name: "MODEL_ID"
    value: "google/gemma-4-26B-A4B-it"
  }
  ports {
    container_port: 8080
  }
  predict_route: "/generate"
  health_route: "/ping"
}
dedicated_resources {
  machine_spec {
    machine_type: "a3-highgpu-1g"
    accelerator_type: NVIDIA_H100_80GB
    accelerator_count: 1
  }
  max_replica_count: 1
}
d

In [ ]:
# @title Vertex AI - Deployment Script

"""
Recommendation: NVIDIA RTX Pro 6000 (Blackwell) with VRAM	96 GB
We prefer NVIDIA L4 (Ada Lovelace) with VRAM 24 GB. Capacity: The L4 has 24 GB of GDDR6 memory.
The Problem: The Gemma 4 26B model in full precision (BF16) requires roughly 52 GB of VRAM just to load the weights. It will not fit on a single L4 in that state.
The Solution (Quantization): To run this on an L4, you must use 4-bit quantization (like AWQ or INT4). In 4-bit mode, the weights take up only ~15–18 GB, which fits comfortably on a single L4.
The Catch: You will likely need to reduce your context window significantly (e.g., from 256K down to 8K or 16K) to stay within the L4's 24GB limit.
"""

!pip install --upgrade google-cloud-aiplatform
#gcloud auth application-default login

"""
import vertexai
from vertexai import model_garden

vertexai.init(project="lunar-352813", location="europe-west4")

model = model_garden.OpenModel("google/gemma4@gemma-4-26b-a4b-it")
endpoint = model.deploy(
  accept_eula=True,
  machine_type="g4-standard-48",
  accelerator_type="NVIDIA_RTX_PRO_6000",
  accelerator_count=1,
  serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
  endpoint_display_name="gemma-4-26b-a4b-it-mg-one-click-deploy",
  model_display_name="gemma-4-26b-a4b-it-1777568163447",
  use_dedicated_endpoint=True,
  reservation_affinity_type="NO_RESERVATION",
)
"""

import vertexai
from vertexai import model_garden

vertexai.init(project="lunar-352813", location="europe-west4")

model = model_garden.OpenModel("google/gemma4@gemma-4-31b-it")
endpoint = model.deploy(
  accept_eula=True,
  machine_type="g4-standard-48",
  accelerator_type="NVIDIA_RTX_PRO_6000",
  accelerator_count=1,
  serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
  endpoint_display_name="gemma-4-31b-it-mg-one-click-deploy",
  model_display_name="gemma-4-31b-it-1777570278133",
  use_dedicated_endpoint=True,
  reservation_affinity_type="NO_RESERVATION",
)